入射角$\theta_s$から探査機角度$\alpha$を導く式
$$
\alpha = 2\theta_s - \sin^{-1} \{ \frac{R}{R+H}\times \sin{\theta_s}\}
$$
から、$\alpha=0.5,1.5,2.5 \dots 179.5$に対応する$\theta_s$をscipy.optimizeで導出

In [16]:
import numpy as np
from scipy import optimize
import xarray as xr

def get_default_param(target):

    match target:
        case "moon":
            H_obs = 100e3       # 観測者の高度[m] (Kaguya)
            D_moon = 1*1e3      # 表層から地下構造までのレゴリスリス層(第一層)の厚さ[m]
            R_moon = 1737400.0  # 月の半径[m]
            e1 = 4.0            # レゴリス層(第一層)の比誘電率
            e2 = 8.0            # レゴリス層の下の地下構造(第二層)の比誘電率
            tandelta = 0.0125   # レゴリス層(第一層)の損失角

        case "ganymede":
            H_obs = 500e3       # 観測者の高度[m] (JUICE)
            D_moon = 1*1e3      # 表層から地下構造までのレゴリスリス層(第一層)の厚さ[m]
            R_moon = 5268000.0/2.0  # ガニメデの半径[m]
            e1 = 3.0            # 第一層の比誘電率
            e2 = 87.0           # 第二層の比誘電率
            tandelta = 0.0      # 第一層の損失角
    
    return e1, e2, H_obs, D_moon, R_moon, e1, e2, tandelta

def calc_alpha(ts, H, R):
    alpha = 2 * np.radians(ts) - np.arcsin(R/(R+H) * np.sin(np.radians(ts)))
    return np.degrees(alpha)

def calc_alpha_opt(ts, H, R, alpha_target):
    alpha = calc_alpha(ts, H, R)
    return alpha - alpha_target

In [17]:
match_alpha = np.arange(0.5,180,1)

hs = [1200,2000,5000,10000]

target = "ganymede"
e1, e2, H_obs, D_moon, R_moon, e1, e2, tandelta = get_default_param(target)

In [22]:
deriv_theta = np.zeros((len(match_alpha), len(hs)))

# hs (altitudes) x match_alpha (alpha targets)
for j, h in enumerate(hs):
    for i, ma in enumerate(match_alpha):
        d_th = optimize.fsolve(calc_alpha_opt, 1.0, args=(h, 1000, ma))
        deriv_theta[i, j] = float(d_th[0])

dth_da = xr.DataArray(deriv_theta, coords={"alpha": match_alpha, "H": hs}, dims=["alpha", "H"])
dth_da.name = "theta_s"
dth_da

<xarray.DataArray 'theta_s' (alpha: 180, H: 4)> Size: 6kB
array([[  0.32352901,   0.29999976,   0.27272718,   0.26190472],
       [  0.9705774 ,   0.89999342,   0.81817936,   0.78571312],
       [  1.61759691,   1.49996954,   1.36362499,   1.30951843],
       [  2.26456829,   2.09991642,   1.90905969,   1.83331856],
       [  2.91147229,   2.69982238,   2.45447911,   2.35711146],
       [  3.5582897 ,   3.29967574,   2.99987887,   2.88089506],
       [  4.20500129,   3.89946482,   3.54525463,   3.4046673 ],
       [  4.85158787,   4.49917798,   4.09060201,   3.9284261 ],
       [  5.49803029,   5.09880357,   4.63591667,   4.45216942],
       [  6.14430939,   5.69832997,   5.18119426,   4.97589519],
       [  6.79040607,   6.29774557,   5.72643043,   5.49960135],
       [  7.43630125,   6.8970388 ,   6.27162085,   6.02328585],
       [  8.0819759 ,   7.49619808,   6.81676119,   6.54694664],
       [  8.72741101,   8.09521189,   7.36184713,   7.07058166],
       [  9.37258765,   8.69406872,   7.90687434,   7.59418886],
       [ 10.01748691,   9.2927571 ,   8.45183854,   8.11776622],
       [ 10.66208994,   9.89126559,   8.99673541,   8.64131167],
       [ 11.30637796,  10.48958278,   9.54156067,   9.16482319],
       [ 11.95033224,  11.0876973 ,  10.08631005,   9.68829874],
       [ 12.5939341 ,  11.68559782,  10.63097927,  10.2117363 ],
...
       [ 93.73677374,  89.98561   ,  85.02882006,  82.83754769],
       [ 94.22807311,  90.48524708,  85.53232087,  83.34030879],
       [ 94.71832475,  90.98411632,  86.03544677,  83.84286894],
       [ 95.20753656,  91.48222137,  86.53819835,  84.34522818],
       [ 95.69571649,  91.97956592,  87.04057619,  84.84738655],
       [ 96.18287251,  92.47615373,  87.54258094,  85.34934413],
       [ 96.66901262,  92.97198857,  88.04421326,  85.85110101],
       [ 97.15414485,  93.4670743 ,  88.54547384,  86.35265729],
       [ 97.63827726,  93.96141478,  89.0463634 ,  86.85401308],
       [ 98.12141791,  94.45501395,  89.54688269,  87.35516853],
       [ 98.6035749 ,  94.94787578,  90.04703248,  87.85612377],
       [ 99.08475633,  95.44000426,  90.54681359,  88.35687897],
       [ 99.56497032,  95.93140343,  91.04622684,  88.85743432],
       [100.04422499,  96.42207739,  91.54527309,  89.35779001],
       [100.52252847,  96.91203025,  92.04395323,  89.85794625],
       [100.9998889 ,  97.40126614,  92.54226817,  90.35790326],
       [101.47631442,  97.88978927,  93.04021885,  90.8576613 ],
       [101.95181317,  98.37760384,  93.53780624,  91.3572206 ],
       [102.42639328,  98.86471409,  94.03503132,  91.85658146],
       [102.90006289,  99.3511243 ,  94.53189511,  92.35574414]])
Coordinates:
  * alpha    (alpha) float64 1kB 0.5 1.5 2.5 3.5 4.5 ... 176.5 177.5 178.5 179.5
  * H        (H) int64 32B 1200 2000 5000 10000

In [23]:
dth_da.to_netcdf("./nc_underground/alpha_to_theta_forhist.nc")